# Stage 3-1. Activation Functions

이 노트북은 `src/nn/activations.py`에 구현된 4종 활성화 함수를 실습한다.

| 함수 | 수식 | 사용 위치 |
|---|---|---|
| `sigmoid` | $\sigma(x) = \frac{1}{1+e^{-x}}$ | hidden layer (binary task) |
| `softmax` | $\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$ | output layer (multiclass) |
| `relu` | $\text{ReLU}(x) = \max(0, x)$ | hidden layer (일반 목적) |
| `identity` | $f(x) = x$ | output layer (regression) |

**학습 목표**
1. 4종 함수의 입출력 값 범위와 형태를 확인한다.
2. 1D 입력에 대한 함수 그래프를 시각화한다.
3. 2D 배치 입력에서 softmax의 동작(행별 정규화)을 확인한다.
4. sigmoid의 수치 안정성 구현(`pos/neg` 분기)을 이해한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.nn.activations import sigmoid, softmax, relu, identity

## 1. 4종 함수 그래프

x 범위 [-6, 6]에서 각 활성화 함수의 출력을 비교한다.

In [ ]:
x = np.linspace(-6, 6, 500)

funcs = [
    (sigmoid,  "sigmoid",  "steelblue",    r"$\sigma(x) = \frac{1}{1+e^{-x}}$",  "[0, 1]"),
    (softmax,  "softmax",  "#E87B4C",       r"row-wise $\frac{e^{x_i}}{\sum e^{x_j}}$", "[0, 1], sum=1"),
    (relu,     "relu",     "mediumseagreen",r"$\max(0, x)$",                        "[0, ∞)"),
    (identity, "identity", "mediumpurple",  r"$x$",                                 "(-∞, ∞)"),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("Activation Functions", fontsize=13, fontweight="bold")

for ax, (fn, name, color, formula, out_range) in zip(axes, funcs):
    if name == "softmax":
        # softmax는 2D 입력 필요 → 1행 배열로 처리
        y = softmax(x.reshape(1, -1)).flatten()
    else:
        y = fn(x)

    ax.plot(x, y, color=color, linewidth=2)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"{name}\n{formula}", fontsize=9)
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.text(0.05, 0.92, f"output: {out_range}", transform=ax.transAxes,
            fontsize=8, color="gray")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 입출력 값 범위 확인

In [ ]:
x_1d = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
print("x       :", x_1d)
print()
print("sigmoid :", sigmoid(x_1d))
print("relu    :", relu(x_1d))
print("identity:", identity(x_1d))

In [ ]:
# softmax: 2D 배치 입력 (N, C)
x_2d = np.array([
    [1.0, 2.0, 3.0],
    [0.0, 0.0, 0.0],
    [10.0, 1.0, 0.1],
])

out = softmax(x_2d)
print("softmax 입력 (3, 3):")
print(x_2d)
print()
print("softmax 출력 (3, 3):")
print(np.round(out, 4))
print()
print("행별 합계 (각 행의 확률 합):")
print(out.sum(axis=1))  # 모두 1.0

> **핵심**: `softmax`는 행(batch) 방향이 아닌 클래스 방향(`axis=-1`)으로 정규화한다.  
> 각 샘플의 출력 합은 항상 1.0이 된다.

## 3. sigmoid 수치 안정성

표준 구현 $\frac{1}{1+e^{-x}}$은 $x \ll 0$일 때 `exp(-x)`가 overflow를 일으킬 수 있다.  
이 프로젝트의 `sigmoid`는 양수/음수를 분기하여 두 가지 동치 형식을 사용한다.

- $x \geq 0$: $\frac{1}{1+e^{-x}}$ (항상 안전)
- $x < 0$: $\frac{e^x}{1+e^x}$ (overflow 방지)

In [ ]:
# 극단값에서 수치 안정성 확인
x_extreme = np.array([-1000.0, -100.0, 0.0, 100.0, 1000.0])

out = sigmoid(x_extreme)
print("x      :", x_extreme)
print("sigmoid:", out)
print("nan 없음:", not np.any(np.isnan(out)))

## 4. MLP에서의 활성화 함수 역할

이 프로젝트의 MLP는 hidden layer에 `sigmoid`를 사용하고, output layer 활성화는 loss 함수 내부에서 처리한다.

| task | hidden activation | output activation (loss 내부) |
|---|---|---|
| multiclass | sigmoid | softmax (cross_entropy 내부) |
| binary | sigmoid | sigmoid (binary_cross_entropy 내부) |
| regression | sigmoid | identity (mse 내부) |

In [ ]:
# hidden layer 통과 후 출력 범위 확인
rng = np.random.default_rng(42)
x_hidden = rng.normal(0, 1, size=(4, 256))  # (batch=4, hidden=256)

out_sigmoid = sigmoid(x_hidden)
out_relu    = relu(x_hidden)

print("sigmoid hidden output:")
print(f"  min={out_sigmoid.min():.4f}, max={out_sigmoid.max():.4f}, mean={out_sigmoid.mean():.4f}")
print()
print("relu hidden output:")
print(f"  min={out_relu.min():.4f}, max={out_relu.max():.4f}, mean={out_relu.mean():.4f}")
print(f"  zero ratio: {np.mean(out_relu == 0):.2%}")

## 5. 정리

| 함수 | 출력 범위 | 특징 |
|---|---|---|
| `sigmoid` | (0, 1) | 양수/음수 분기로 수치 안정성 확보 |
| `softmax` | (0, 1), 행합=1 | 2D 배치 입력, `axis=-1` 방향 정규화 |
| `relu` | [0, ∞) | 음수 출력 0, 계산 효율 우수 |
| `identity` | (-∞, ∞) | 변환 없음, regression output에 사용 |

**핵심 설계 원칙**
- 이 모듈의 활성화 함수는 **forward 전용**이다. backward(gradient)는 `layers.py`의 레이어 클래스가 담당한다.
- `softmax`와 loss gradient는 수치 안정성을 위해 loss 함수 내부에서 결합하여 처리한다.